# 00. Preparar bases — bronze Censo + CPF

Importa parquets bronze no DuckDB, filtra por UF, limpa variáveis, empilha `registro_unificado` e gera `ground_truth_clusters` a partir de `cohort_dedup`.


In [ ]:
import sys
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from config import (
    CENSO_ESPECIE2_ARQUIVO, CENSO_PESSOAS_ARQUIVO, CPF_ARQUIVO,
    COHORT_DEDUP_ARQUIVO, FILTRO_UF, OUTPUT_DIR,
    CENSO_COL_ENDERECO, CENSO_COL_FACE, CENSO_COL_ID_DOMICILIO,
    CENSO_COL_ID_MORADOR, CENSO_COL_NOME_MAE, CENSO_COL_PRIMEIRO_NOME,
    CENSO_COL_SEQ_ESPECIE, CENSO_COL_SETOR, CENSO_COL_SEXO, CENSO_COL_SOBRENOME,
    CPF_COL_CEP, CPF_COL_CPF, CPF_COL_DATA_NASC, CPF_COL_ESTADO,
    CPF_COL_NOME, CPF_COL_NOME_MAE, CPF_COL_SEXO, CPF_COL_UF,
    ESPECIE2_COL_CEP, ESPECIE2_COL_ENDERECO, ESPECIE2_COL_FACE,
    ESPECIE2_COL_QUADRA, ESPECIE2_COL_SEQ, ESPECIE2_COL_SETOR,
    benchmark_checkpoint, censo_dob_sql, censo_uf_sql, cpf_norm_sql,
    cep_norm_sql, export_parquet, get_connection, normalize_date_sql,
    print_paths, require_input, sql_optional_col, uf_filter_clause,
)
from features import featurize_three_part_names_batch

print_paths()
for label, p in [
    ('CPF', CPF_ARQUIVO), ('CENSO_PESSOAS', CENSO_PESSOAS_ARQUIVO),
    ('CENSO_ESPECIE2', CENSO_ESPECIE2_ARQUIVO), ('COHORT', COHORT_DEDUP_ARQUIVO),
]:
    require_input(p, label=label)

con = get_connection()
print('FILTRO_UF ativo:', FILTRO_UF)


## 1. Inspecionar bronze


In [ ]:
for label, path in [
    ('cpf', CPF_ARQUIVO),
    ('censo_pessoas', CENSO_PESSOAS_ARQUIVO),
    ('censo_especie2', CENSO_ESPECIE2_ARQUIVO),
]:
    print(f'\n=== {label} ===')
    display(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{path}') LIMIT 0").df())


## 2. Importar bronze


In [ ]:
for tbl in [
    'cpf_bronze_raw', 'censo_pessoas_raw', 'censo_especie2_raw', 'cohort_dedup_raw',
    'cpf_filtrado', 'censo_pessoas_filtrado', 'censo_especie2_filtrado',
    'censo_endereco', 'cpf_staging', 'cpf_feat', 'cpf_registros',
    'censo_staging', 'censo_feat', 'censo_registros',
    'registro_unificado', 'ground_truth_pairs', 'ground_truth_clusters',
]:
    con.execute(f'DROP TABLE IF EXISTS {tbl}')

con.execute(f"CREATE OR REPLACE TABLE cpf_bronze_raw AS SELECT * FROM read_parquet('{CPF_ARQUIVO}')")
con.execute(f"CREATE OR REPLACE TABLE censo_pessoas_raw AS SELECT * FROM read_parquet('{CENSO_PESSOAS_ARQUIVO}')")
con.execute(f"CREATE OR REPLACE TABLE censo_especie2_raw AS SELECT * FROM read_parquet('{CENSO_ESPECIE2_ARQUIVO}')")
con.execute(f"CREATE OR REPLACE TABLE cohort_dedup_raw AS SELECT * FROM read_parquet('{COHORT_DEDUP_ARQUIVO}')")

for t in ['cpf_bronze_raw', 'censo_pessoas_raw', 'censo_especie2_raw', 'cohort_dedup_raw']:
    benchmark_checkpoint(con, t, f'SELECT COUNT(*) FROM {t}')


## 3. Filtrar por UF


In [ ]:
CENSO_UF = censo_uf_sql('p.' + CENSO_COL_SETOR)
CPF_UF_EXPR = sql_optional_col('c', CPF_COL_UF) if CPF_COL_UF else 'NULL'
CPF_UF_FILTER = uf_filter_clause(CPF_UF_EXPR, FILTRO_UF) if CPF_COL_UF else 'TRUE'

con.execute(f'''
CREATE OR REPLACE TABLE cpf_filtrado AS
SELECT c.* FROM cpf_bronze_raw c WHERE {CPF_UF_FILTER}
''')

con.execute(f'''
CREATE OR REPLACE TABLE censo_pessoas_filtrado AS
SELECT p.* FROM censo_pessoas_raw p
WHERE {uf_filter_clause(CENSO_UF, FILTRO_UF)}
''')

con.execute(f'''
CREATE OR REPLACE TABLE censo_especie2_filtrado AS
SELECT e.* FROM censo_especie2_raw e
WHERE {uf_filter_clause(censo_uf_sql('e.' + ESPECIE2_COL_SETOR), FILTRO_UF)}
''')

for t in ['cpf_filtrado', 'censo_pessoas_filtrado', 'censo_especie2_filtrado']:
    benchmark_checkpoint(con, t, f'SELECT COUNT(*) FROM {t}')


## 4. Join Censo + espécie2


In [ ]:
cep_col = f'e."{ESPECIE2_COL_CEP}"' if ESPECIE2_COL_CEP else 'NULL'
cep_norm = cep_norm_sql(cep_col) if ESPECIE2_COL_CEP else "''"

con.execute(f'''
CREATE OR REPLACE TABLE censo_endereco AS
SELECT DISTINCT
    CAST(p.{CENSO_COL_ID_MORADOR} AS VARCHAR) AS person_id_censo,
    {cep_norm} AS cep_domicilio
FROM censo_pessoas_filtrado p
LEFT JOIN censo_especie2_filtrado e
    ON p.{CENSO_COL_SETOR} = e.{ESPECIE2_COL_SETOR}
   AND p.{CENSO_COL_QUADRA} = e.{ESPECIE2_COL_QUADRA}
   AND p.{CENSO_COL_FACE} = e.{ESPECIE2_COL_FACE}
   AND p.{CENSO_COL_ENDERECO} = e.{ESPECIE2_COL_ENDERECO}
   AND p.{CENSO_COL_SEQ_ESPECIE} = e.{ESPECIE2_COL_SEQ}
''')


## 5. CPF — limpeza e nomes


In [ ]:
CPF_N = cpf_norm_sql(f'c."{CPF_COL_CPF}"')
DT_NASC = normalize_date_sql(f'c."{CPF_COL_DATA_NASC}"')
NOME_MAE = sql_optional_col('c', CPF_COL_NOME_MAE)
SEXO = sql_optional_col('c', CPF_COL_SEXO)
CEP = cep_norm_sql(sql_optional_col('c', CPF_COL_CEP)) if CPF_COL_CEP else "''"
ESTADO = sql_optional_col('c', CPF_COL_ESTADO)
UF_COL = sql_optional_col('c', CPF_COL_UF) if CPF_COL_UF else 'NULL'

con.execute(f'''
CREATE OR REPLACE TABLE cpf_staging AS
SELECT
    {CPF_N} AS cpf_norm,
    TRIM(CAST(c."{CPF_COL_NOME}" AS VARCHAR)) AS nome_completo_raw,
    {DT_NASC} AS data_nascimento,
    {NOME_MAE} AS nome_mae_raw,
    {SEXO} AS sexo_raw,
    {CEP} AS cep,
    {ESTADO} AS estado,
    {UF_COL} AS uf
FROM cpf_filtrado c
WHERE {CPF_N} IS NOT NULL
''')

featurize_three_part_names_batch(
    con, source_table='cpf_staging', target_table='cpf_feat', name_col='nome_completo_raw',
)

con.execute('''
CREATE OR REPLACE TABLE cpf_registros AS
SELECT
    'cpf_' || cpf_norm AS unique_id, 'cpf' AS origem, cpf_norm,
    nome_completo_norm AS nome_completo, primeiro_nome, nome_meio, ultimo_nome,
    nome_completo_phon, primeiro_nome_phon, ultimo_nome_phon,
    data_nascimento,
    UPPER(TRIM(CAST(nome_mae_raw AS VARCHAR))) AS nome_mae,
    UPPER(TRIM(CAST(sexo_raw AS VARCHAR))) AS sexo,
    cep, CAST(estado AS VARCHAR) AS estado, CAST(uf AS VARCHAR) AS uf,
    CAST(NULL AS VARCHAR) AS person_id_censo, CAST(NULL AS VARCHAR) AS id_domicilio
FROM cpf_feat
''')


## 6. Censo — limpeza e nomes


In [ ]:
DT_PESSOA = censo_dob_sql()
DT_NASC_C = normalize_date_sql(DT_PESSOA)
NOME_COMPLETO = f"TRIM(COALESCE(CAST(p.{CENSO_COL_PRIMEIRO_NOME} AS VARCHAR), '') || ' ' || COALESCE(CAST(p.{CENSO_COL_SOBRENOME} AS VARCHAR), ''))"
NOME_MAE_C = sql_optional_col('p', CENSO_COL_NOME_MAE)
SEXO_C = sql_optional_col('p', CENSO_COL_SEXO)
UF_C = censo_uf_sql('p.' + CENSO_COL_SETOR)

con.execute(f'''
CREATE OR REPLACE TABLE censo_staging AS
SELECT
    CAST(p.{CENSO_COL_ID_MORADOR} AS VARCHAR) AS person_id_censo,
    CAST(p.{CENSO_COL_ID_DOMICILIO} AS VARCHAR) AS id_domicilio,
    {NOME_COMPLETO} AS nome_completo_raw,
    {DT_NASC_C} AS data_nascimento,
    {NOME_MAE_C} AS nome_mae_raw,
    {SEXO_C} AS sexo_raw,
    {UF_C} AS uf,
    COALESCE(e.cep_domicilio, '') AS cep
FROM censo_pessoas_filtrado p
LEFT JOIN censo_endereco e ON CAST(p.{CENSO_COL_ID_MORADOR} AS VARCHAR) = e.person_id_censo
''')

featurize_three_part_names_batch(
    con, source_table='censo_staging', target_table='censo_feat', name_col='nome_completo_raw',
)

con.execute('''
CREATE OR REPLACE TABLE censo_registros AS
SELECT
    'censo_' || person_id_censo AS unique_id, 'censo' AS origem,
    CAST(NULL AS VARCHAR) AS cpf_norm,
    nome_completo_norm AS nome_completo, primeiro_nome, nome_meio, ultimo_nome,
    nome_completo_phon, primeiro_nome_phon, ultimo_nome_phon,
    data_nascimento,
    UPPER(TRIM(CAST(nome_mae_raw AS VARCHAR))) AS nome_mae,
    UPPER(TRIM(CAST(sexo_raw AS VARCHAR))) AS sexo,
    cep, CAST(NULL AS VARCHAR) AS estado, CAST(uf AS VARCHAR) AS uf,
    person_id_censo, id_domicilio
FROM censo_feat
WHERE person_id_censo IS NOT NULL
''')


## 7. Stack + ground truth


In [ ]:
con.execute('''
CREATE OR REPLACE TABLE registro_unificado AS
SELECT * FROM censo_registros UNION ALL SELECT * FROM cpf_registros
''')

CPF_GT = cpf_norm_sql('CPF_NORM')
con.execute(f'''
CREATE OR REPLACE TABLE ground_truth_pairs AS
SELECT DISTINCT
    'censo_' || CAST(PERSON_ID_CENSO AS VARCHAR) AS unique_id_censo,
    'cpf_' || {CPF_GT} AS unique_id_cpf,
    CAST(PERSON_ID_CENSO AS VARCHAR) AS person_id_censo,
    {CPF_GT} AS cpf_norm
FROM cohort_dedup_raw
WHERE PERSON_ID_CENSO IS NOT NULL AND CPF_NORM IS NOT NULL
''')

con.execute('''
CREATE OR REPLACE TABLE ground_truth_clusters AS
WITH pairs AS (
    SELECT unique_id_censo, unique_id_cpf,
           'gt_' || person_id_censo || '_' || cpf_norm AS cluster_id
    FROM ground_truth_pairs
),
labeled AS (
    SELECT unique_id_censo AS unique_id, cluster_id FROM pairs
    UNION ALL
    SELECT unique_id_cpf AS unique_id, cluster_id FROM pairs
)
SELECT r.unique_id, COALESCE(l.cluster_id, r.unique_id) AS cluster
FROM registro_unificado r
LEFT JOIN labeled l ON r.unique_id = l.unique_id
''')

benchmark_checkpoint(con, 'registro_unificado', 'SELECT COUNT(*) FROM registro_unificado')


## 8. Export


In [ ]:
p1 = export_parquet(con, 'registro_unificado', path=OUTPUT_DIR / 'registro_unificado.parquet')
p2 = export_parquet(con, 'ground_truth_clusters', path=OUTPUT_DIR / 'ground_truth_clusters.parquet')
print('Exportado:', p1, p2)
con.close()
